# Is there anything about kidney cancer already?

*Standalone -- no prior vignette required.*

I'm a grad student. I don't have a disease picked out yet, and I don't have much budget to go collect new data myself -- so before I commit to anything, I just want to know: is there *already* data out there about kidney cancer? I don't know what columns CDA has, I don't know what words the data uses internally, and honestly I don't want to find out before I even know if this is worth my time.

So I'm not going to go look anything up first. I'm just going to type the word and see what happens.

In [ ]:
from cdapython import *
from itables import init_notebook_mode, show
init_notebook_mode(all_interactive=True)
import itables.options as opt
opt.classes="display nowrap compact"
opt.buttons=["copyHtml5", "csvHtml5", "excelHtml5"]
opt.maxBytes=0
opt.columnDefs = [{"className": "dt-left", "targets": "_all"}]

In [ ]:
summarize_subjects('kidney')

46,569 subjects, from one word, with zero setup. I didn't tell it which column to look in -- `kidney` could be an anatomic site, a diagnosis, part of a file description, anything -- it just searched everywhere at once and found every subject connected to the term in any way.

I don't know yet if that's useful to me. Let's see what these subjects actually look like before deciding anything. The summary doesn't show every column by default, so let me check what's available and pull in a few that sound relevant:

In [ ]:
columns()

In [ ]:
summarize_subjects('kidney', add_columns=['anatomic_site', 'diagnosis', 'format'])

Interesting -- a lot of these subjects also show up with `vcf` in their file formats. If there's sequencing data sitting around too, that's worth knowing before I pick a direction. I can just add a second word -- I don't need to abandon the first one or restart with a more careful query:

In [ ]:
summarize_subjects('kidney', 'vcf', add_columns=['anatomic_site', 'diagnosis', 'format'])

Two free-text terms like this means "must touch both" -- every subject here is connected to *both* kidney and vcf somewhere in their record, still searched across every column at once. 2,516 subjects, with 23,096 vcf files between them -- a much smaller, more specific number than my first search, and I got there without writing a single filter string.

Now I want to actually look at the `anatomic_site` column on these results, and there's more going on in it than I expected. The biggest single value isn't "kidney" at all -- it's a missing/null value, which just means plenty of these subjects matched on something other than anatomy (their diagnosis, a file description, whatever). After that, "kidney" itself is the largest real value, which makes sense.

But then "blood" shows up with over 20,000 hits. That has nothing to do with kidney anatomy. What's actually happening is that global search found a subject through *some* connection to "kidney," and then showed me *everything* about that subject -- including a completely unrelated blood draw that has nothing to do with why they matched in the first place. That's not a mistake; it's the whole point of searching at the subject level instead of the field level. I didn't ask for blood data, but now I know it's there, which I never would have found if I'd only searched inside the anatomic_site field directly.

The second thing I notice: "right kidney" and "left kidney" also show up, each more specific than the plain word I typed. That one *is* an anatomy match -- CDA's anatomic_site field is backed by a real medical ontology (UBERON), and searching a broad term like "kidney" automatically pulls in anything more specific that sits underneath it in that hierarchy. "Right kidney" and "left kidney" are two of those more-specific terms; there are dozens more sitting even deeper in that same branch that never happened to show up directly in my particular result set. I didn't need to know that hierarchy existed, let alone look anything up in it, for this to work.

So two different things just happened in the same result: one subject-level ("here's everything about this person, including blood work you didn't ask for") and one anatomy-specific ("here's a more precise version of the term you typed, found automatically"). If I want to know *why* a specific anatomic_site row matched -- the literal word, or something more specific rolling up -- I can ask CDA to show its work:

In [ ]:
summarize_subjects('kidney', 'vcf', add_columns=['anatomic_site', 'diagnosis', 'format'], add_extras=['all'])

Same subjects, same count -- `add_extras` doesn't change what matched, it just annotates *why*. A row with "right kidney" now shows that it matched because it rolls up to "kidney" in the ontology.

Worth knowing: this kind of automatic rollup currently only happens for `anatomic_site` and `disease` -- those are the two fields backed by a real external ontology (UBERON and ICD-O-3, respectively). Most other harmonized fields, like `sex` or `species`, are fully standardized too, but don't have a hierarchy to roll up through in the first place -- there's no such thing as a "more specific" sex. A few fields (`stage`, `treatment_type`, and others) aren't harmonized yet at all. So `add_extras` won't do anything interesting on those -- not because something's broken, but because there's nothing underneath them yet to show.

I still haven't written a single filter, looked up a controlled vocabulary, or checked what table anything lives in. 2,516 subjects with both kidney involvement and sequencing data is enough for me to decide this is worth pursuing -- and if I do, [Getting oriented before I filter](02_systematic_browsing.ipynb) is where I'd go next to actually get oriented in what's here before building a real query.